# Barranquilla MOPAC: paso de tiempo y enlaces X–H de la región QM

En la corrida anterior (`qmmm_guess2_a100`) la deriva de la tripsina con tolerancias 0,001 y 0,0001 venía de unos
pocos saltos de 1–3 kJ/mol. En el Mac se vio de dónde salen: con pasos de 1 fs, velocity Verlet comete ese error en
un paso cuando un átomo de la región QM se mueve rápido (HD1 de His57, H de Lys15 y Ala16; los X–H de la región QM
no tienen restricciones); las fuerzas QM/MM son exactas (energía a lo largo del paso en 10 subpasos: +0,015 frente a
+2,18 kJ/mol). Esos errores se acumulan casi como un paseo aleatorio: a 0,5 fs, ~5 veces menos que a 1 fs; con los
X–H restringidos a 1 fs, no menos (en 1,5 ps; los átomos pesados de la región QM también se mueven rápido). Aquí, desde el mismo estado inicial que la corrida anterior, tres semillas
del NVT previo (11, 12, 13) por variante, tolerancia 0,001:

| Corrida | Qué es |
|---|---|
| `flex_dt1_s*` | X–H libres, 1 fs (como la corrida anterior) |
| `flex_dt0.5_s*` | X–H libres, 0,5 fs (lo que usa `run_qmmm_md.py` por defecto) |
| `con_dt1_s*` | X–H de la región QM restringidos (`--qm-constraints`, como `qmshake=1` de Amber), 1 fs |
| `flex_dt0.5_tol0.01` | la configuración por defecto (0,5 fs, tolerancia 0,01), una semilla |

Todas: 0,4 ps de NVT y luego 5 ps de NVE, OpenMM en precisión `mixed`, MOPAC en la GPU. Unos 90 min en total.

**Antes de empezar:** *Entorno de ejecución → Cambiar tipo de entorno → GPU A100*, y luego *Ejecutar todo*. Los
resultados se guardan en Google Drive (`MyDrive/mopac_paper_bench/RESULTS_NAME`); si la sesión se corta, vuelva a
ejecutar todo y lo terminado se salta. La última celda descarga un zip.

## 1. Configuración

In [ ]:
COMMIT = '229b6fe4'            # --qm-constraints / --qm-free
RESULTS_NAME = 'qmmm_dt_a100'
PREP_FROM = 'qmmm_guess2_a100'  # estado inicial de la corrida anterior (si no está, se prepara uno nuevo)
USE_DRIVE = True
QUICK = False                  # True: pocas pasos (prueba del notebook)
REQUIRE_GPU = 'A100'           # None: aceptar cualquier GPU

## 2. GPU, CPU y carpeta de resultados

In [ ]:
import os, re, subprocess, sys, time, shutil, json
from pathlib import Path

def run(cmd, check=True, env=None, log=None):
    # Run a shell command and stream its output to the cell (and to a log file)
    print('$', cmd, flush=True)
    full = dict(os.environ); full.update(env or {})
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=full)
    fh = open(log, 'w') if log else None
    for line in p.stdout:
        print(line, end='', flush=True)
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    if check and p.returncode != 0:
        raise RuntimeError(f'exit {p.returncode}: {cmd}')
    return p.returncode

run('nvidia-smi')
gpu = subprocess.run('nvidia-smi --query-gpu=name,compute_cap --format=csv,noheader', shell=True,
                     capture_output=True, text=True).stdout.strip()
cpu = subprocess.run("lscpu | grep 'Model name' | sed 's/.*: *//'", shell=True, capture_output=True,
                     text=True).stdout.strip()
NCPU = os.cpu_count()
print(f'GPU: {gpu}\nCPU: {cpu}, {NCPU} núcleos lógicos')
CC = gpu.split(',')[-1].strip().replace('.', '')
if REQUIRE_GPU and REQUIRE_GPU not in gpu:
    raise RuntimeError(f'Esta sesión tiene {gpu}, no una {REQUIRE_GPU}. Cambie el tipo de entorno a {REQUIRE_GPU} '
                     f'(o ponga REQUIRE_GPU = None) y vuelva a ejecutar todo.')

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/mopac_paper_bench')
else:
    BASE = Path('/content/mopac_paper_bench')
OUT = BASE / RESULTS_NAME
OUT.mkdir(parents=True, exist_ok=True)
(OUT / 'env.json').write_text(json.dumps({'gpu': gpu, 'cpu': cpu, 'ncpu': NCPU, 'commit': COMMIT}, indent=1))
print('resultados en', OUT)

## 3. Compilar MOPAC (biblioteca) con OpenBLAS

~10 minutos.

In [ ]:
SRC, BUILD = Path('/content/mopac_gpu'), Path('/content/build')
run('apt-get update -qq && DEBIAN_FRONTEND=noninteractive apt-get install -y -qq gfortran ninja-build '
    'libopenblas-dev liblapack-dev > /dev/null')
os.environ['PATH'] = '/usr/local/cuda/bin:' + os.environ['PATH']
shutil.rmtree(SRC, ignore_errors=True); shutil.rmtree(BUILD, ignore_errors=True)
run(f'git clone -q https://github.com/juvenalyosa/mopac_gpu.git {SRC}')
run(f'git -C {SRC} checkout -q {COMMIT}')
run(f'git -C {SRC} log -1 --format="%h %ad %s" --date=short')
head = subprocess.run(f'git -C {SRC} rev-parse HEAD', shell=True, capture_output=True, text=True).stdout.strip()
if not head.startswith(COMMIT):
    raise SystemExit(f'HEAD {head} no es COMMIT {COMMIT}')
stamp = OUT / 'commit.txt'
if stamp.exists() and not head.startswith(stamp.read_text().strip()[:8]):
    raise SystemExit(f'{OUT} tiene resultados de {stamp.read_text().strip()}; use otro RESULTS_NAME')
stamp.write_text(head + '\n')
t0 = time.time()
run(f'cmake -S {SRC} -B {BUILD} -G Ninja -DGPU=ON -DTESTS=OFF -DCMAKE_BUILD_TYPE=Release -DCUDA_ARCHS={CC} '
    f'-DENABLE_MKL=OFF -DBLA_VENDOR=OpenBLAS > /content/cmake.log 2>&1 || (tail -40 /content/cmake.log; exit 1)')
run(f'cmake --build {BUILD} --target mopac --parallel {NCPU} > /content/build.log 2>&1 '
    f'|| (grep -i -B2 -A6 error /content/build.log | head -80; exit 1)')
LIBMOPAC = next(BUILD.glob('libmopac.so*'))
print(f'compilado en {time.time() - t0:.0f} s: {LIBMOPAC}')

## 4. OpenMM y CuPy

In [ ]:
m = re.search(r'release (\d+)', subprocess.run('nvcc --version', shell=True, capture_output=True, text=True).stdout)
cupy_pkg = f'cupy-cuda{m.group(1) if m else 12}x'
other = [p for p in subprocess.run('pip list 2>/dev/null', shell=True, capture_output=True, text=True).stdout.split()
         if p.startswith('cupy-cuda') and p != cupy_pkg]
if other:
    run('pip uninstall -y -q ' + ' '.join(other), check=False)
run(f"pip install -q 'openmm[cuda12]==8.6.1' {cupy_pkg} 2>&1 | tail -2", check=False)
run(f"{sys.executable} -c \"import openmm as m; print('OpenMM', m.__version__, "
    f"[m.Platform.getPlatform(i).getName() for i in range(m.Platform.getNumPlatforms())])\"", check=False)
run(f"cd {SRC}/scripts/qmmm && {sys.executable} -c \"from qmmm_pme import gpu_available; "
    f"print('QM-PME grid on the GPU (CuPy):', gpu_available())\"", check=False)

## 5. Corridas NVE (`scripts/qmmm/nve_drift.py`)

El estado inicial es el de `PREP_FROM` (copiado), así que estas corridas empiezan donde empezaron las de 1 fs con
X–H libres de la corrida anterior. Cada corrida: NVT de 0,4 ps y NVE de 5 ps con el mismo paso; lectura de energía
cada 10 fs. Una corrida que falla queda registrada y las demás siguen.

In [ ]:
import urllib.request
ND = SRC / 'scripts' / 'qmmm' / 'nve_drift.py'
dC = OUT / 'runs' / 'C_drift'
prep = dC / 'prep'
src_prep = BASE / PREP_FROM / 'runs' / 'C_drift' / 'prep'
if not (prep / 'state.xml').exists():
    prep.mkdir(parents=True, exist_ok=True)
    if (src_prep / 'state.xml').exists():
        for f in ('state.xml', 'top.pdb'):
            shutil.copy2(src_prep / f, prep / f)
        (prep / 'from.txt').write_text(str(src_prep) + '\n')
        print('estado inicial copiado de', src_prep)
    else:
        pdb = OUT / 'runs' / '_pdb' / '2PTC.pdb'
        pdb.parent.mkdir(parents=True, exist_ok=True)
        if not pdb.exists():
            urllib.request.urlretrieve('https://files.rcsb.org/download/2PTC.pdb', pdb)
        print('no está', src_prep, '; se prepara un estado nuevo (50 ps de MM)')
        run(f'{sys.executable} -u {ND} prepare {pdb} --out {prep} --padding 1.0 --variants E:57=HID '
            f'--mm-equil-ps {10 if QUICK else 50} --platform CUDA --precision mixed', log=prep / 'log.txt')
QMSEL = '--qm-residues E:57:sc,E:102:sc,E:195:sc,I:15,I:16 --ron 10 --roff 12 --precision mixed'
CON = '--qm-constraints'
runs = [(f'{v}_s{seed}', f'{o} --tolerance 0.001', dt, seed) for seed in (11, 12, 13)
        for v, o, dt in (('flex_dt1', '', 1.0), ('flex_dt0.5', '', 0.5), ('con_dt1', CON, 1.0))]
runs.append(('flex_dt0.5_tol0.01', '--tolerance 0.01', 0.5, 11))
failed = []
for name, opts, dt, seed in runs:
    d = dC / name
    nvt, nve, every = (int(round(x / dt)) for x in ((20, 200, 10) if QUICK else (400, 5000, 10)))
    cmd = (f'{sys.executable} -u {ND} run --prep {prep} --out {d} --libmopac {LIBMOPAC} {QMSEL} {opts} '
           f'--platform CUDA --dt {dt} --nvt {nvt} --nve {nve} --every {every} --seed {seed}')
    if (d / 'summary.json').exists() and (d / 'cmd.txt').exists() and (d / 'cmd.txt').read_text() == cmd:
        print(name, 'ya hecho'); continue
    d.mkdir(parents=True, exist_ok=True)
    (d / 'cmd.txt').write_text(cmd)
    (d / 'summary.json').unlink(missing_ok=True)
    env = dict(os.environ, OMP_NUM_THREADS=str(NCPU), OPENBLAS_NUM_THREADS=str(NCPU), MOPAC_FORCEGPU='1')
    t0 = time.time()
    with open(d / 'log.txt', 'w') as fh:
        rc = subprocess.run(cmd, shell=True, env=env, stdout=fh, stderr=subprocess.STDOUT).returncode
    tail = (d / 'log.txt').read_text(errors='ignore').strip().splitlines()
    print(name, 'rc', rc, f'{time.time() - t0:.0f} s', tail[-1][:300] if tail else '')
    if rc != 0:
        failed.append(name)
        print('    FALLÓ; final del registro:\n' + '\n'.join(tail[-15:]))
print('fallidas:', failed or 'ninguna')

## 6. Resultados

`slope_qm_nonconservative`: la deriva de E_qm + W (energía de la fuerza QM/MM más su trabajo); `inc_sd`: desviación
de sus incrementos de 10 fs; `block_sd`: desviación de sus cambios en bloques de 0,5 ps (cuánto se acumula);
`sum_top10` / `sum_without_top10`: los diez mayores incrementos y lo que queda sin ellos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
dr = []
for f in sorted(dC.glob('*/summary.json')):
    s = json.loads(f.read_text()); s['name'] = f.parent.name
    r = np.loadtxt(f.parent / 'nve.txt')
    x = np.diff((r[:, 4] - r[0, 4]) + (r[:, 5] - r[0, 5]))
    top = np.argsort(-np.abs(x))[:10]
    nb = len(x) // 50 * 50
    s.update(inc_median=float(np.median(x)), inc_sd=float(np.std(x)), block_sd=float(x[:nb].reshape(-1, 50).sum(1).std()),
             sum_top10=float(x[top].sum()), sum_without_top10=float(x.sum() - x[top].sum()))
    dr.append(s)
dr_df = pd.DataFrame(dr)
dr_df.to_csv(OUT / 'qmmm_dt_drift.csv', index=False)
display(dr_df[[c for c in ['name', 'dt', 'tolerance', 'slope_total', 'slope_total_se', 'slope_qm_nonconservative',
                           'slope_qm_nonconservative_se', 'slope_rest', 'slope_rest_se', 'inc_median', 'inc_sd', 'block_sd',
                           'sum_top10', 'sum_without_top10', 'scf_iterations_nve', 'nvt_s_per_step', 'nve_s_per_step']
               if c in dr_df]])
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for f in sorted(dC.glob('*/nve.txt')):
    r = np.loadtxt(f)
    ax[0].plot(r[:, 0], r[:, 1] - r[0, 1], lw=0.7, label=f.parent.name)
    ax[1].plot(r[:, 0], (r[:, 4] - r[0, 4]) + (r[:, 5] - r[0, 5]), lw=0.8, label=f.parent.name)
ax[0].set_title('E total − E(0)'); ax[1].set_title('fuerza QM/MM: E_qm + W')
for a_ in ax:
    a_.set_xlabel('ps'); a_.set_ylabel('kJ/mol'); a_.legend(fontsize=7); a_.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(OUT / 'qmmm_dt_drift.png', dpi=150); plt.show()

## 7. Descargar

In [ ]:
tmp = Path('/content/_qmmm_dt') / RESULTS_NAME
shutil.rmtree(tmp.parent, ignore_errors=True)
tmp.mkdir(parents=True)
for f in ('qmmm_dt_drift.csv', 'qmmm_dt_drift.png', 'env.json', 'commit.txt'):
    if (OUT / f).exists():
        shutil.copy2(OUT / f, tmp / f)
for pat in ('summary.json', 'log.txt', 'cmd.txt', 'nve.txt', 'from.txt'):
    for f in (OUT / 'runs').rglob(pat):
        if '_pdb' in f.parts:
            continue
        dst = tmp / 'runs' / f.relative_to(OUT / 'runs')
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dst)
z = shutil.make_archive(f'/content/{RESULTS_NAME}', 'zip', tmp.parent)
print(z, f'{os.path.getsize(z) / 1e6:.1f} MB')
from google.colab import files
files.download(z)